[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch12-benchmarking/04-nang-luong-va-nghiem-thu-ban-nen.ipynb)

# Năng lượng, và nghiệm thu một bản nén: INT8 tiết kiệm gì, và con số tổng che giấu gì

Chương 12 dành một chiều riêng cho năng lượng, vì một con số công suất chỉ có nghĩa khi biết nó được đọc ở
điểm vận hành nào, trong ranh giới đo nào. Rồi chương quay lại MobileNetV2 lượng tử hóa (quantization)
sang INT8: mô hình năng lượng 45 nm nói nó tiết kiệm 6.6 lần, còn bảng 18 nói độ chính xác tổng gần như
không đổi trong khi hiệu chuẩn và các trường hợp biên thì xấu đi.

Sổ tay này tính lại các con số năng lượng của chương, đếm lại MobileNetV2 từ torchvision, rồi tự làm một
giao thức nghiệm thu kiểu bảng 18 trên một mạng FashionMNIST thật: đặt ngưỡng trước, nén, rồi đo.

**Bạn sẽ làm:**
1. đổi công suất và thời gian ra năng lượng của một lần suy luận;
2. tính công suất trung bình theo chu kỳ hoạt động (duty cycle), và kiểm ba tuyên bố về công suất của sách;
3. đặt tám thiết bị của bảng 14 lên một trục, và tính phụ trội của hạ tầng;
4. dựng lại phép tính nhẩm 1.5 với số đếm của MobileNetV2 thật, rồi thêm phần mô hình đơn giản hoá bỏ qua;
5. nghiệm thu một bản INT8 và một bản INT4 theo ngưỡng đặt trước: độ chính xác, hiệu chuẩn và lát cắt.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Năng lượng là công suất nhân thời gian

Sách đưa một đồng nhất thức bền vững cho năng lượng mỗi truy vấn: $E = P \times T$. Công suất ở đây là
công suất trung bình **đo được** trong cửa sổ suy luận; công suất thiết kế nhiệt (Thermal Design Power,
TDP) ghi trên tờ thông số không thay được phép đo ấy.

**Dự đoán:** một lần suy luận 10 ms ở 300 W tốn bao nhiêu oát giờ?

In [ ]:
CONG_SUAT = sach(300, nguon="300 W · 10 ms · 3 J")  # W
t_ngan = sach(10, nguon="300 W · 10 ms · 3 J") / 1000  # giây
t_dai = sach(100, nguon="100 ms · ≈0.0083 Wh", tol=0.5) / 1000
theo_sach("năng lượng một lần suy luận 10 ms (J)", CONG_SUAT * t_ngan, sach=3, nguon="300 W · 10 ms · 3 J")
theo_sach("đổi ra Wh", CONG_SUAT * t_ngan / 3600, sach=0.0008, nguon="3 J · ≈0.0008 Wh", tol=0.00005)
theo_sach("một lần suy luận 100 ms (Wh)", CONG_SUAT * t_dai / 3600, sach=0.0083, nguon="100 ms · ≈0.0083 Wh",
          tol=0.00005)

gpt3 = sach(1_287, nguon="≈1,287 MWh", tol=0.5) * 1e6 * 3600  # J
print(f"năng lượng ước tính để huấn luyện GPT-3 bằng {gpt3 / (CONG_SUAT * t_ngan):.2e} lần suy luận 10 ms như trên")

Phép chia cuối là của sổ tay, không phải của sách, và chỉ để cảm nhận cỡ: năng lượng huấn luyện được trả
một lần rồi chia cho cả đời mô hình, còn năng lượng suy luận cộng dồn theo từng truy vấn, nên ở quy mô đủ
lớn nó trở thành chi phí vận hành chính.

## 2. Công suất trung bình theo chu kỳ hoạt động

Thời lượng pin không phụ thuộc công suất tức thời mà phụ thuộc tổng năng lượng qua một mẫu sử dụng thật. Gọi
$\alpha$ là phần thời gian hệ thống thực sự chạy suy luận:

$$P_{\text{avg}} = \alpha P_{\text{active}} + (1 - \alpha) P_{\text{idle}}$$

Góc nhìn 1.7 của sách đặt một thiết bị biên tiêu thụ 2 W khi hoạt động, 50 mW khi nghỉ, và chạy suy luận 1
phần trăm thời gian.

**Dự đoán:** công suất hiệu dụng gần 2 W hay gần 50 mW hơn?

In [ ]:
P_HOAT_DONG = sach(2, nguon="50 mW · 2 W · 1 phần trăm")  # W
P_NGHI = sach(50, nguon="50 mW · 2 W · 1 phần trăm") / 1000  # W
ALPHA = sach(1, nguon="50 mW · 2 W · 1 phần trăm") / 100
theo_sach("α", ALPHA, sach=0.01, nguon="0.01 · 0.99")
theo_sach("1 − α", 1 - ALPHA, sach=0.99, nguon="0.01 · 0.99")


def cong_suat_tb(alpha, p_hd=P_HOAT_DONG, p_nghi=P_NGHI):
    return alpha * p_hd + (1 - alpha) * p_nghi


theo_sach("công suất hiệu dụng (mW)", 1000 * cong_suat_tb(ALPHA), sach=69.5, nguon="1 phần trăm · ≈69.5 mW", tol=0.05)
alpha_bang = P_NGHI / (P_HOAT_DONG + P_NGHI)  # α P_hd = (1 − α) P_nghi
print(f"phần hoạt động và phần nghỉ tốn năng lượng bằng nhau khi α = {100 * alpha_bang:.2f} %")
PIN_WH = 3.7  # giả định của sổ tay: một pin 1,000 mAh ở 3.7 V
print(f"pin {PIN_WH} Wh (giả định): {PIN_WH / cong_suat_tb(ALPHA):.0f} giờ ở công suất hiệu dụng,"
      f" {PIN_WH / P_HOAT_DONG:.2f} giờ nếu tin con số 2 W")

cac_alpha = np.logspace(-4, 0, 200)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.loglog(100 * cac_alpha, 1000 * cong_suat_tb(cac_alpha), color="#2a78d6", lw=2, label="công suất trung bình")
ax.loglog(100 * cac_alpha, 1000 * cac_alpha * P_HOAT_DONG, "--", color="#eb6834", lw=1.5, label="phần hoạt động")
ax.loglog(100 * cac_alpha, 1000 * (1 - cac_alpha) * P_NGHI, ":", color="#1baf7a", lw=2, label="phần nghỉ")
ax.plot(100 * ALPHA, 1000 * cong_suat_tb(ALPHA), "ko", ms=6, zorder=5)
ax.annotate("sách: 1 %, ≈69.5 mW", (100 * ALPHA, 1000 * cong_suat_tb(ALPHA)), xytext=(-10, 10),
            textcoords="offset points", fontsize=8, ha="right")
ax.axvline(100 * alpha_bang, color="0.5", lw=1, ls="-.")
ax.text(100 * alpha_bang * 1.15, 1.5, f"hai phần bằng nhau: {100 * alpha_bang:.2f} %", fontsize=8, color="0.3")
ax.set_xlabel("α: phần thời gian chạy suy luận (%)")
ax.set_ylabel("mW")
ax.set_title("Thiết bị 2 W khi hoạt động, 50 mW khi nghỉ (kịch bản của sách)")
ax.set_ylim(0.1, 4_000)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Dưới khoảng 2.4 phần trăm, phần lớn năng lượng là năng lượng **nghỉ**, nên giảm công suất chờ có lợi hơn
tăng tốc suy luận. Trên ngưỡng ấy thì ngược lại. Đó là lý do sách tách camera chuông cửa, gần như luôn nghỉ,
khỏi chuỗi phân tích video thời gian thực, gần như luôn suy luận.

### Ba tuyên bố về công suất

Cùng góc nhìn 1.7 còn nói một thiết bị quảng cáo 45 TOPS ở đỉnh có thể chỉ duy trì khoảng 20 TOPS trong một
lần chạy nhiệt kéo dài, nên phải đo với tải duy trì hơn 30 giây. Phần đo công suất mở đầu bằng một ví dụ
khác: 10 TOPS ở 0.5 W trên tờ thông số, 3 TOPS ở 2 W khi giảm tốc do nhiệt (throttling). Bảng 13 thì đặt
MobileNetV2 INT8 trên Cortex-M7 cạnh EdgeTPU, trong một kịch bản minh hoạ mà sách ghi rõ không phải phép đo
có kiểm soát.

In [ ]:
dinh_tops = sach(45, nguon="45 · ≈20 TOPS", tol=0.5)
ben_tops = sach(20, nguon="45 · ≈20 TOPS", tol=0.5)
chay_toi_thieu = sach(30, nguon="≈20 TOPS · 30 giây", tol=0.5)
print(f"chạy nhiệt kéo dài: {100 * ben_tops / dinh_tops:.0f} % mức đỉnh; benchmark nên chạy quá {chay_toi_thieu} giây")

qc = sach(10, nguon="10 TOPS ở 0.5 W") / sach(0.5, nguon="10 TOPS ở 0.5 W")  # TOPS/W trên tờ thông số
tt = sach(3, nguon="3 TOPS ở 2 W") / sach(2, nguon="3 TOPS ở 2 W")  # TOPS/W khi chạy thật
print(f"tờ thông số: {qc:.1f} TOPS/W, khi chạy thật: {tt:.1f} TOPS/W")
theo_sach("khoảng cách hiệu quả", qc / tt, sach=13.3, nguon="3 TOPS ở 2 W · 13.3×")

t_m7, t_tpu = sach(15, nguon="≈15 · ≈2 ms", tol=0.5), sach(2, nguon="≈15 · ≈2 ms", tol=0.5)  # ms suy luận
dc_m7, dc_tpu = sach(18, nguon="7.5× · ≈18 · ≈6 ms", tol=0.5), sach(6, nguon="7.5× · ≈18 · ≈6 ms", tol=0.5)
p_m7, p_tpu = sach(120, nguon="≈120 · ≈500 mW", tol=0.5), sach(500, nguon="≈120 · ≈500 mW", tol=0.5)  # mW
theo_sach("tăng tốc phần suy luận", t_m7 / t_tpu, sach=7.5, nguon="≈2 ms · 7.5×", tol=0.05)
theo_sach("tăng tốc đầu cuối", dc_m7 / dc_tpu, sach=3, nguon="≈6 ms · ≈3×", tol=0.05)
theo_sach("công suất EdgeTPU gấp", p_tpu / p_m7, sach=4.2, nguon="≈500 mW · ≈4.2×", tol=0.05)
theo_sach("năng lượng một lần suy luận, Cortex-M7 (mJ)", p_m7 * t_m7 / 1000, sach=1.8, nguon="≈4.2× · ≈1.8", tol=0.05)
theo_sach("năng lượng một lần suy luận, EdgeTPU (mJ)", p_tpu * t_tpu / 1000, sach=1, nguon="≈1.8 · ≈1 mJ", tol=0.05)
theo_sach("EdgeTPU tiết kiệm năng lượng", (p_m7 * t_m7) / (p_tpu * t_tpu), sach=1.8, nguon="≈1 mJ · ≈1.8×", tol=0.05)
print(f"phần ngoài suy luận: {dc_m7 - t_m7:g} ms trên đường Cortex-M7, {dc_tpu - t_tpu:g} ms trên đường EdgeTPU")

Kiểm ba điều. Hai đầu của một tỉ số TOPS/W phải đọc ở **cùng** một điểm vận hành: tử số ở batch tốt nhất và
mẫu số ở lúc gần nghỉ thì mô tả một trạng thái chip không bao giờ ở. EdgeTPU tốn công suất gấp hơn 4 lần
nhưng xong nhanh hơn 7.5 lần, nên năng lượng mỗi lần suy luận vẫn thấp hơn; con số ấy chỉ tính phần suy
luận, không tính công suất ngủ hay năng lượng đánh thức.

Phần ngoài suy luận của bảng 13 không khớp thành một khoản tiền xử lý chung: 3 ms trên đường Cortex-M7, 4 ms
trên đường EdgeTPU. Sách chỉ nói tiền xử lý chạy trên CPU ở cả hai đường, và không giải thích phần chênh 1
ms. Một cách đọc, chỉ là giả định của sổ tay, là thời gian chuyển dữ liệu qua lại bộ tăng tốc. Định luật
Amdahl ở sổ tay 02 nói vì sao 7.5 lần ở phần suy luận chỉ còn 3 lần đầu cuối.

Ngộ nhận cuối chương đặt một hệ thống 1,200 truy vấn mỗi giây ở 4.2 W cạnh một hệ thống 1,000 truy vấn mỗi
giây ở 1.8 W:

In [ ]:
q_nhanh, w_nhanh = sach(1_200, nguon="1,200 · 4.2 W"), sach(4.2, nguon="1,200 · 4.2 W")
q_cham, w_cham = sach(1_000, nguon="1.8 W · 1,000"), sach(1.8, nguon="4.2 W · 1.8 W")
theo_sach("công suất gấp", w_nhanh / w_cham, sach=2.3, nguon="1,000 · 2.3×", tol=0.05)
theo_sach("thông lượng mất khi chọn hệ chậm (%)", 100 * (1 - q_cham / q_nhanh), sach=17, nguon="2.3× · 17 phần trăm",
          tol=0.5)
for ten, q, w in (("hệ nhanh", q_nhanh, w_nhanh), ("hệ chậm", q_cham, w_cham)):
    print(f"{ten}: {1000 * w / q:.2f} mJ mỗi truy vấn, {q / w:.0f} truy vấn mỗi jun")

## 3. Gần mười một bậc độ lớn

Bảng 14 xếp tám điểm triển khai từ 150 µW tới cả một tủ máy chủ 10 kW, và hình 6 kéo hai đầu ra xa hơn: 5.6
µW ở sàn siêu nhỏ, khoảng 498 kW ở trần một cụm huấn luyện. Không một dụng cụ đo nào phủ được cả dải ấy.

In [ ]:
BANG14 = {  # (thấp, cao), oát
    "bộ xử lý quyết định thần kinh (NDP)": (sach(150, nguon="150 µW · 25 mW") * 1e-6,) * 2,
    "vi điều khiển M7": (sach(25, nguon="150 µW · 25 mW") * 1e-3,) * 2,
    "Raspberry Pi 4": (sach(3.5, nguon="25 mW · 3.5 W"),) * 2,
    "điện thoại thông minh": (sach(4, nguon="3.5 W · 4 W"),) * 2,
    "camera thông minh": (sach(10, nguon="4 W · 10–15 W"), sach(15, nguon="4 W · 10–15 W")),
    "máy chủ biên": (sach(65, nguon="10–15 W · 65–95 W"), sach(95, nguon="10–15 W · 65–95 W")),
    "nút máy chủ học máy": (sach(300, nguon="65–95 W · 300–500 W"), sach(500, nguon="65–95 W · 300–500 W")),
    "tủ máy chủ học máy": (sach(4, nguon="300–500 W · 4–10 kW") * 1e3, sach(10, nguon="300–500 W · 4–10 kW") * 1e3),
}
san = sach(5.6, nguon="5.6 µW · ≈498 kW", tol=0.05) * 1e-6
tran = sach(498, nguon="5.6 µW · ≈498 kW", tol=0.5) * 1e3
bac_bang = math.log10(BANG14["tủ máy chủ học máy"][1] / BANG14["bộ xử lý quyết định thần kinh (NDP)"][0])
print(f"bảng 14, từ 150 µW tới 10 kW: {bac_bang:.2f} bậc độ lớn (sách: gần tám)")
print(f"hình 6, từ 5.6 µW tới 498 kW: {math.log10(tran / san):.2f} bậc độ lớn (sách: gần mười một)")

fig, ax = plt.subplots(figsize=(8, 3.8))
for j, (ten, (thap, cao)) in enumerate(BANG14.items()):
    if thap == cao:  # sách cho một con số: vẽ một điểm
        ax.plot([thap], [j], "o", color="#2a78d6", ms=8)
    else:  # sách cho một khoảng: vẽ một vạch
        ax.plot([thap, cao], [j, j], color="#2a78d6", lw=8, solid_capstyle="round")
    if cao < 100:  # chữ bên phải điểm nhỏ, bên trái điểm lớn, để không chạm đường 498 kW
        ax.text(cao * 2.5, j, ten, va="center", fontsize=8)
    else:
        ax.text(thap / 2.5, j, ten, va="center", ha="right", fontsize=8)
for gt, chu in ((san, "5.6 µW, sàn siêu nhỏ (hình 6)"), (tran, "≈498 kW, cụm huấn luyện (hình 6)")):
    ax.axvline(gt, color="#eb6834", lw=1, ls="--")
    ax.text(gt * 1.5 if gt < 1 else gt / 1.5, len(BANG14) - 0.3, chu, fontsize=8, color="#eb6834",
            ha="left" if gt < 1 else "right")
ax.set_xscale("log")
ax.set_xlim(1e-6, 2e6)
ax.set_ylim(-0.7, len(BANG14) + 0.2)
ax.set_yticks([])
ax.set_xlabel("công suất (W, thang log)")
ax.set_title("Bảng 14: tám điểm triển khai trên một trục")
plt.tight_layout()
plt.show()

Ở trung tâm dữ liệu, làm mát và cấp điện là hạ tầng dùng chung, khó gán cho một tải học máy. Sách nói riêng
làm mát có thể chiếm 20 tới 30 phần trăm tổng công suất của cơ sở, và chỉ số hiệu quả sử dụng điện (Power
Usage Effectiveness, PUE), tức tổng năng lượng của cơ sở chia cho năng lượng của thiết bị CNTT, đi từ 1.1 ở
cơ sở hiệu quả nhất tới hơn 2.0 ở cơ sở kém tối ưu.

In [ ]:
pue_tot, pue_kem = sach(1.1, nguon="20–25 °C · 1.1 · hơn 2.0"), sach(2.0, nguon="1.1 · hơn 2.0")
for pue in (pue_tot, pue_kem):
    print(f"PUE {pue}: mỗi 1 MWh cho máy tính, cơ sở tiêu thụ {pue:.1f} MWh;"
          f" phần ngoài máy tính chiếm {100 * (1 - 1 / pue):.0f} % tổng")
lm_thap = sach(20, nguon="20–30 phần trăm · 20–25 °C") / 100
lm_cao = sach(30, nguon="20–30 phần trăm · 20–25 °C") / 100
print(f"nếu làm mát là phụ trội duy nhất, 20 tới 30 % tổng tương ứng PUE {1 / (1 - lm_thap):.2f} tới {1 / (1 - lm_cao):.2f}")
nhan_doi = sach(1.5, nguon="≈1.5 năm", tol=0.05)
print(f"số phép tính mỗi kWh gấp đôi mỗi {nhan_doi} năm, tức khoảng {2 ** (10 / nhan_doi):.0f} lần mỗi mười năm")

## 4. Vì sao INT8 tiết kiệm năng lượng

Phép tính nhẩm 1.5 dùng mô hình năng lượng thành phần ở 45 nm của Horowitz. Bảng 15 cho năng lượng một phép
nhân theo độ chính xác số, bảng 16 cho năng lượng mỗi byte theo tầng bộ nhớ. Sách lưu ý giá trị tuyệt đối
lẫn tỉ lệ đều tuỳ thiết kế mạch và tiến trình.

In [ ]:
NHAN = {  # pJ mỗi phép nhân
    "FP32": sach(3.7, nguon="≈3.7 · ≈1.1 · ≈0.2 pJ", tol=0.05),
    "FP16": sach(1.1, nguon="≈3.7 · ≈1.1 · ≈0.2 pJ", tol=0.05),
    "INT8": sach(0.2, nguon="≈3.7 · ≈1.1 · ≈0.2 pJ", tol=0.05),
}
theo_sach("FP16 so với FP32", NHAN["FP16"] / NHAN["FP32"], sach=0.3, nguon="0.3× · 0.05×", tol=0.005)
theo_sach("INT8 so với FP32", NHAN["INT8"] / NHAN["FP32"], sach=0.05, nguon="0.3× · 0.05×", tol=0.005)
theo_sach("FP32 đắt hơn INT8", NHAN["FP32"] / NHAN["INT8"], sach=18, nguon="0.05× · ≈18×", tol=0.6)
BYTE = {  # pJ mỗi byte
    "thanh ghi": sach(0.1, nguon="≈0.1 · ≈1 · ≈5", tol=0.005),
    "L1": sach(1, nguon="≈0.1 · ≈1 · ≈5", tol=0.05),
    "L2": sach(5, nguon="≈0.1 · ≈1 · ≈5", tol=0.5),
    "DRAM": sach(160, nguon="≈160 pJ mỗi byte", tol=0.5),
}
theo_sach("L1 so với thanh ghi", BYTE["L1"] / BYTE["thanh ghi"], sach=10, nguon="10× · 50×")
theo_sach("L2 so với thanh ghi", BYTE["L2"] / BYTE["thanh ghi"], sach=50, nguon="10× · 50×")
theo_sach("DRAM so với thanh ghi", BYTE["DRAM"] / BYTE["thanh ghi"], sach=1_600, nguon="50× · 1,600×")

Đọc một byte từ DRAM tốn hơn 1,600 lần một lần đọc thanh ghi, và gấp hơn 40 lần một phép nhân FP32. Vì vậy
trong suy luận, truy cập bộ nhớ ngoài chip thường chi phối ngân sách năng lượng.

### Đếm MobileNetV2

Sách đặt MobileNetV2 ở 14 MB FP32, 3.5 MB INT8 và 600 MFLOP. Ô dưới dựng MobileNetV2 của torchvision, với
trọng số ngẫu nhiên vì chỉ cần đếm, rồi gắn một hàm móc (hook) vào mọi tầng tích chập và tuyến tính để đếm
phép nhân-cộng ở ảnh 224×224, cùng số giá trị kích hoạt các tầng ấy ghi ra.

**Dự đoán:** với FP32, phần đọc trọng số hay phần số học tốn năng lượng hơn?

In [ ]:
import torch.nn as nn
import torchvision

mb = torchvision.models.mobilenet_v2(weights=None).eval()
n_tham_so = sum(p.numel() for p in mb.parameters())
dem = {"nhan_cong": 0, "kich_hoat": 0}


def moc(m, vao, ra):
    dem["kich_hoat"] += ra.numel()
    if isinstance(m, nn.Conv2d):  # mỗi giá trị ra cần (kênh vào / nhóm) × kích thước nhân phép nhân-cộng
        dem["nhan_cong"] += ra.numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
    else:
        dem["nhan_cong"] += ra.numel() * m.in_features


moc_da_gan = [m.register_forward_hook(moc) for m in mb.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
with torch.no_grad():
    mb(torch.zeros(1, 3, 224, 224))
for h in moc_da_gan:
    h.remove()
FLOP_DEM = 2 * dem["nhan_cong"]
print(f"tham số: {n_tham_so:,}; phép nhân-cộng: {dem['nhan_cong']:,}; giá trị kích hoạt: {dem['kich_hoat']:,}")
theo_sach("tham số (triệu)", n_tham_so / 1e6, sach=3.5, nguon="≈72 · 3.5 triệu", tol=0.05)
theo_sach("FP32 (MB)", 4 * n_tham_so / 1e6, sach=14, nguon="14 MB → 3.5 MB", tol=0.05)
theo_sach("INT8 (MB)", n_tham_so / 1e6, sach=3.5, nguon="14 MB → 3.5 MB", tol=0.01)
theo_sach("MFLOP mỗi lần suy luận", FLOP_DEM / 1e6, sach=600, nguon="120 µJ · 600 MFLOP", tol=2)

Số đếm cho 601.5 MFLOP, còn sách viết 600: sách làm tròn, và phần tính dưới đây dùng đúng 600 của sách để
khớp bảng 17. Giờ là bảng 17: một lần đọc DRAM cho mỗi trọng số, cộng năng lượng phép nhân cho mỗi FLOP.

In [ ]:
FLOP_SACH = sach(600, nguon="120 µJ · 600 MFLOP") * 1e6
PJ = 1e-6  # đổi pJ ra µJ
doc = {k: b * n_tham_so * BYTE["DRAM"] * PJ for k, b in (("FP32", 4), ("INT8", 1))}  # µJ
tinh = {k: FLOP_SACH * NHAN[k] * PJ for k in ("FP32", "INT8")}
theo_sach("đọc trọng số FP32 (µJ)", doc["FP32"], sach=2243, nguon="2243 · 561 µJ")
theo_sach("đọc trọng số INT8 (µJ)", doc["INT8"], sach=561, nguon="2243 · 561 µJ")
theo_sach("số học FP32 (µJ)", tinh["FP32"], sach=2_220, nguon="2,220 · 120 µJ")
theo_sach("số học INT8 (µJ)", tinh["INT8"], sach=120, nguon="2,220 · 120 µJ")
tong = {k: doc[k] + tinh[k] for k in doc}
theo_sach("tổng FP32 (µJ)", tong["FP32"], sach=4_463, nguon="4,463 · 681 µJ")
theo_sach("tổng INT8 (µJ)", tong["INT8"], sach=681, nguon="4,463 · 681 µJ")
theo_sach("tiết kiệm ở phần đọc", doc["FP32"] / doc["INT8"], sach=4, nguon="681 µJ · 4×")
theo_sach("tiết kiệm ở phần số học", tinh["FP32"] / tinh["INT8"], sach=18.5, nguon="4× · 18.5×")
theo_sach("tiết kiệm tổng", tong["FP32"] / tong["INT8"], sach=6.6, nguon="18.5× · 6.6×", tol=0.05)
print(f"với 601.5 MFLOP đếm được, tổng FP32 là {doc['FP32'] + FLOP_DEM * NHAN['FP32'] * PJ:,.0f} µJ")

### Phần mô hình đơn giản hoá bỏ qua

Sách nói rõ bảng 17 bỏ qua lưu lượng kích hoạt, hành vi bộ đệm, phép cộng, phụ trội điều khiển và công suất
tĩnh, nên nó không phải phép đo cả thiết bị. Ô dưới thêm lại **một** khoản, kích hoạt, theo hai **giả định
của sổ tay**: mỗi giá trị kích hoạt được ghi ra một lần và đọc lại một lần, hoặc từ DRAM, hoặc từ bộ đệm L2.
Kích hoạt INT8 cũng nhỏ đi 4 lần như trọng số. Để so, sách trích một phép đo trên TensorFlow Mobile: 57.3
phần trăm năng lượng suy luận dành cho di chuyển dữ liệu.

In [ ]:
di_chuyen_do = sach(57.3, nguon="57.3 phần trăm")
ket_qua_nl = {}
for noi in ("DRAM", "L2"):
    kh = {k: 2 * dem["kich_hoat"] * b * BYTE[noi] * PJ for k, b in (("FP32", 4), ("INT8", 1))}
    ket_qua_nl[noi] = kh
    t32, t8 = tong["FP32"] + kh["FP32"], tong["INT8"] + kh["INT8"]
    phan = 100 * (doc["FP32"] + kh["FP32"]) / t32
    print(f"kích hoạt qua {noi:<4} (giả định): FP32 {t32:,.0f} µJ, INT8 {t8:,.0f} µJ, tiết kiệm {t32 / t8:.1f} lần;"
          f" di chuyển dữ liệu chiếm {phan:.0f} % năng lượng FP32")
print(f"bảng 17 không có kích hoạt: di chuyển dữ liệu chiếm {100 * doc['FP32'] / tong['FP32']:.0f} % năng lượng FP32;"
      f" phép đo sách trích: {di_chuyen_do} %")

fig, ax = plt.subplots(figsize=(8, 3.6))
cot = ["FP32, bảng 17", "INT8, bảng 17", "FP32, + kích hoạt L2", "INT8, + kích hoạt L2",
       "FP32, + kích hoạt DRAM", "INT8, + kích hoạt DRAM"]
gia_tri = [(doc["FP32"], tinh["FP32"], 0), (doc["INT8"], tinh["INT8"], 0),
           (doc["FP32"], tinh["FP32"], ket_qua_nl["L2"]["FP32"]), (doc["INT8"], tinh["INT8"], ket_qua_nl["L2"]["INT8"]),
           (doc["FP32"], tinh["FP32"], ket_qua_nl["DRAM"]["FP32"]),
           (doc["INT8"], tinh["INT8"], ket_qua_nl["DRAM"]["INT8"])]
trai = np.zeros(len(cot))
for j, (ten, mau, gach) in enumerate((("đọc trọng số từ DRAM", "#2a78d6", None), ("số học", "#eb6834", None),
                                       ("kích hoạt (giả định)", "#1baf7a", "//"))):
    rong = np.array([g[j] for g in gia_tri])
    ax.barh(range(len(cot)), rong, left=trai, color=mau, hatch=gach, edgecolor="white", label=ten)
    trai += rong
for j, t in enumerate(trai):
    ax.text(t + 150, j, f"{t:,.0f} µJ", va="center", fontsize=8)
ax.set_yticks(range(len(cot)), cot, fontsize=8)
ax.invert_yaxis()
ax.set_xlim(0, trai.max() * 1.18)
ax.set_xlabel("µJ mỗi lần suy luận, mô hình 45 nm (tính, không đo)")
ax.set_title("MobileNetV2: bảng 17, rồi thêm kích hoạt")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

Kích hoạt ở 224×224 nhiều gấp gần hai lần trọng số. Nếu chúng đi qua DRAM, năng lượng FP32 tăng gần gấp ba, và
mức tiết kiệm co từ 6.6 về 4.6 lần, sát mức 4 lần của phần byte, vì lúc ấy phần lớn năng lượng là byte, mà byte
chỉ nhỏ đi 4 lần. Nếu chúng nằm lại trong bộ đệm L2 thì bảng 17 gần đúng. Hai giả định cho hai câu trả lời
khác nhau, nên đó chính là chỗ sách đòi phép đo có kiểm soát trên cả thiết bị trước mọi tuyên bố về pin.

## 5. Nghiệm thu một bản nén

### Điểm mù của độ chính xác

Trước khi nén, hai phép tính nhỏ cho thấy vì sao một con số độ chính xác không đủ. Sách nêu một mô hình phát
hiện gian lận đạt 99 phần trăm độ chính xác mà chỉ có 10 phần trăm recall, tức chỉ bắt được một trong 10 giao
dịch gian lận. Ô dưới dựng lại một tập như thế với 10,000 giao dịch, 1 phần trăm là gian lận (con số 10,000 là
giả định của sổ tay).

In [ ]:
N_GD = 10_000  # giả định
ti_le = sach(1, nguon="99 · 1 · 99 phần trăm") / 100
gian_lan = round(N_GD * ti_le)
theo_sach("độ chính xác khi luôn đoán không gian lận (%)", 100 * (N_GD - gian_lan) / N_GD, sach=99,
          nguon="99 · 1 · 99 phần trăm")
recall = sach(10, nguon="99 phần trăm · 10 phần trăm recall") / 100
bat_duoc = round(recall * gian_lan)
sai_cho_phep = round(N_GD * (1 - sach(99, nguon="99 phần trăm · 10 phần trăm recall") / 100))
bao_nham = sai_cho_phep - (gian_lan - bat_duoc)  # số lỗi còn lại là báo nhầm
print(f"recall 10 %: bắt {bat_duoc} trên {gian_lan} vụ; độ chính xác 99 % cho phép {sai_cho_phep} lỗi, nên còn"
      f" {bao_nham} lần báo nhầm, precision {100 * bat_duoc / (bat_duoc + bao_nham):.0f} %")

Mô hình không làm gì cũng đạt 99 phần trăm. Đó là lý do sách đòi precision và recall khi lớp mất cân bằng, và
đòi đánh giá tách theo nhóm, vì một mô hình 95 phần trăm tổng thể có thể chỉ đạt 60 phần trăm ở một nhóm then
chốt.

### Giao thức, kiểu bảng 18

Bảng 18 của sách là một giao thức kiểm định INT8 với các giá trị giả định: top-1 từ 71.8 xuống 70.9 phần trăm,
dưới ngưỡng 1 điểm nên đạt, nhưng sai số hiệu chuẩn kỳ vọng (expected calibration error, ECE) từ 0.031 lên
0.089 và độ chính xác trên trường hợp biên từ 68.2 xuống 61.4 phần trăm. Sách dặn đặt ngưỡng **trước** khi nén.

In [ ]:
top1_truoc, top1_sau = sach(71.8, nguon="71.8 → 70.9"), sach(70.9, nguon="71.8 → 70.9")
bien_truoc, bien_sau = sach(68.2, nguon="68.2 → 61.4"), sach(61.4, nguon="68.2 → 61.4")
ece_truoc, ece_sau = sach(0.031, nguon="0.031 → 0.089"), sach(0.089, nguon="0.031 → 0.089")
theo_sach("bảng 18, mức giảm top-1 (điểm)", top1_truoc - top1_sau, sach=0.9, nguon="68.2 → 61.4 · 6.8 · 0.9")
theo_sach("bảng 18, mức giảm trên trường hợp biên (điểm)", bien_truoc - bien_sau, sach=6.8, nguon="68.2 → 61.4 · 6.8")
print(f"bảng 18, ECE tăng {ece_sau / ece_truoc:.1f} lần, từ {ece_truoc} lên {ece_sau}")

# Ngưỡng đặt TRƯỚC khi nén. Top-1 lấy ngưỡng của sách; hai ngưỡng sau là giả định của sổ tay.
NGUONG_TOP1 = sach(1, nguon="0.9 · 1 điểm")  # điểm phần trăm
NGUONG_ECE = 0.01  # giả định: ECE không được tăng quá 0.01 so với bản gốc
NGUONG_BIEN = 1.0  # giả định: mỗi lát cắt trường hợp biên không được giảm quá 1 điểm
NGUONG_TIN = sach(85, nguon="≈5 phần trăm · 85 phần trăm", tol=0.5) / 100  # "chỉ hành động khi độ tự tin trên 85 %"

Ngưỡng top-1 là của sách. Hai ngưỡng còn lại là giả định của sổ tay, vì sách nói ECE không có ngưỡng tốt hay
kém dùng chung, mà phải so với một dung sai riêng của tác vụ và với bản chưa nén.

Mô hình là một mạng 784-256-128-10 trên FashionMNIST, huấn luyện 6 epoch trên 50,000 ảnh, giữ 10,000 ảnh còn lại
để kiểm định. Hai bản nén được làm bằng tay, theo cùng một cách: trọng số lượng tử hóa đối xứng theo từng kênh
ra, đầu vào của mỗi tầng tuyến tính lượng tử hóa đối xứng theo cả tensor, với hệ số co giãn (scale factor)
lấy từ 1,000 ảnh huấn luyện làm tập hiệu chuẩn. Phép tính vẫn chạy bằng số thực trên các giá trị đã làm tròn,
tức lượng tử hóa giả, nên kết quả là kết quả của lưới INT8 hay INT4, còn tốc độ thì không.

In [ ]:
tap_hl = torchvision.datasets.FashionMNIST(DATA, train=True, download=True)
tap_kt = torchvision.datasets.FashionMNIST(DATA, train=False, download=True)
TB, DLC = (tap_hl.data.float() / 255).mean(), (tap_hl.data.float() / 255).std()


def chuan_hoa(anh):
    """Ảnh uint8 hoặc số thực trong [0, 1], cỡ (n, 28, 28) → vectơ đã chuẩn hoá."""
    x = anh.float() / 255 if anh.dtype == torch.uint8 else anh
    return ((x - TB) / DLC).reshape(len(x), -1)


hoan_vi = torch.randperm(60_000, generator=torch.Generator().manual_seed(SEED))
X_hl, Y_hl = chuan_hoa(tap_hl.data[hoan_vi[:50_000]]), tap_hl.targets[hoan_vi[:50_000]]
X_kd, Y_kd = chuan_hoa(tap_hl.data[hoan_vi[50_000:]]), tap_hl.targets[hoan_vi[50_000:]]
X_kt, Y_kt = chuan_hoa(tap_kt.data), tap_kt.targets

dat_hat_giong()
goc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
toi_uu = torch.optim.Adam(goc.parameters(), lr=1e-3)
sinh = torch.Generator().manual_seed(SEED)
for _ in range(6):
    thu_tu = torch.randperm(len(X_hl), generator=sinh)
    for i in range(0, len(X_hl), 128):
        chon = thu_tu[i:i + 128]
        toi_uu.zero_grad()
        nn.functional.cross_entropy(goc(X_hl[chon]), Y_hl[chon]).backward()
        toi_uu.step()
goc.eval()


class LuongTu(nn.Module):
    """Bản lượng tử hóa giả của một mạng Sequential: trọng số theo kênh, đầu vào theo tensor, đối xứng."""

    def __init__(self, mang, hieu_chuan, bit):
        super().__init__()
        self.mang, self.q = mang, 2 ** (bit - 1) - 1
        self.he_so_w, self.he_so_x = [], []
        h = hieu_chuan
        for m in mang:
            if isinstance(m, nn.Linear):
                self.he_so_w.append(m.weight.abs().amax(1, keepdim=True) / self.q)
                self.he_so_x.append(h.abs().max() / self.q)
            h = m(h)

    def lam_tron(self, x, he_so):
        return torch.clamp(torch.round(x / he_so), -self.q, self.q) * he_so

    def forward(self, x):
        k = 0
        for m in self.mang:
            if isinstance(m, nn.Linear):
                x = self.lam_tron(x, self.he_so_x[k]) @ self.lam_tron(m.weight, self.he_so_w[k]).T + m.bias
                k += 1
            else:
                x = m(x)
        return x


with torch.no_grad():
    MO_HINH = {"FP32": goc, "INT8": LuongTu(goc, X_hl[:1_000], 8), "INT4": LuongTu(goc, X_hl[:1_000], 4)}
print({ten: f"{100 * (m(X_kt).argmax(1) == Y_kt).float().mean():.2f} %" for ten, m in MO_HINH.items()})

Ba lát cắt trường hợp biên mô phỏng định nghĩa của sách trên 2,000 ảnh kiểm tra: **xoay** hơn 30 độ (sổ tay lấy
30 tới 45 độ), **che** hơn 50 phần trăm ảnh (16 trên 28 hàng hoặc cột, từ một cạnh chọn ngẫu nhiên), và **tối**,
độ sáng còn 25 phần trăm, một giả định của sổ tay thay cho "ánh sáng dưới 100 lux".

ECE được đo bằng một ước lượng cố định: 15 khoảng độ tự tin đều nhau, như sách dặn phải cố định cách ước lượng.
Co giãn nhiệt độ (temperature scaling) chia logit cho một hằng số $T$, chọn để cực tiểu hoá mất mát trên tập
kiểm định, không phải trên tập kiểm tra.

In [ ]:
import torchvision.transforms.functional as TF

rng = np.random.default_rng(SEED)
chon_bien = rng.choice(len(tap_kt), 2_000, replace=False)
anh_bien, Y_bien = tap_kt.data[chon_bien].float() / 255, tap_kt.targets[chon_bien]
goc_xoay = rng.uniform(30, 45, 2_000) * rng.choice([-1, 1], 2_000)
xoay = torch.stack([TF.rotate(a[None], float(g))[0] for a, g in zip(anh_bien, goc_xoay)])
che = anh_bien.clone()
for k, canh in enumerate(rng.integers(0, 4, 2_000)):  # che 16 trên 28 hàng hoặc cột, từ một cạnh
    if canh == 0:
        che[k, :16] = 0
    elif canh == 1:
        che[k, -16:] = 0
    elif canh == 2:
        che[k, :, :16] = 0
    else:
        che[k, :, -16:] = 0
LAT_CAT = {"xoay 30–45°": xoay, "che hơn 50 %": che, "tối, 25 % độ sáng": anh_bien * 0.25}


def ece(logit, y, T=1.0, so_khoang=15):
    """Sai số hiệu chuẩn kỳ vọng với các khoảng độ tự tin đều nhau: ước lượng cố định cho cả giao thức."""
    tin, doan = torch.softmax(logit / T, 1).max(1)
    dung = (doan == y).float()
    tong_ = 0.0
    for a, b in zip(torch.linspace(0, 1, so_khoang + 1)[:-1], torch.linspace(0, 1, so_khoang + 1)[1:]):
        trong = (tin > a) & (tin <= b)
        if trong.any():
            tong_ += trong.float().mean().item() * abs(tin[trong].mean().item() - dung[trong].mean().item())
    return tong_


def chon_T(logit, y):
    cac_T = torch.linspace(0.5, 3, 251)
    return float(cac_T[int(np.argmin([nn.functional.cross_entropy(logit / t, y).item() for t in cac_T]))])


TEN_LOP = tap_kt.classes
do = {}
with torch.no_grad():
    for ten, m in MO_HINH.items():
        logit = m(X_kt)
        doan = logit.argmax(1)
        theo_lop = [(doan[Y_kt == c] == c).float().mean().item() for c in range(10)]
        T = chon_T(m(X_kd), Y_kd)
        do[ten] = {
            "top-1 (%)": 100 * (doan == Y_kt).float().mean().item(),
            "lớp kém nhất (%)": 100 * min(theo_lop),
            "ECE": ece(logit, Y_kt),
            "T": T,
            "ECE sau co giãn": ece(logit, Y_kt, T),
        }
        for luc, T_ in (("trước co giãn", 1.0), ("sau co giãn", T)):  # quy tắc "chỉ hành động khi độ tự tin trên 85 %"
            tu_tin = torch.softmax(logit / T_, 1).max(1).values > NGUONG_TIN
            do[ten][f"tự động hoá, {luc} (%)"] = 100 * tu_tin.float().mean().item()
            do[ten][f"đúng khi tự động, {luc} (%)"] = 100 * (doan[tu_tin] == Y_kt[tu_tin]).float().mean().item()
        for lc, a in LAT_CAT.items():
            do[ten][f"{lc} (%)"] = 100 * (m(chuan_hoa(a)).argmax(1) == Y_bien).float().mean().item()
        if ten == "FP32":
            lop_kem = TEN_LOP[int(np.argmin(theo_lop))]

print(f"{'chỉ số':<38}" + "".join(f"{ten:>10}" for ten in do))
for chi_so in do["FP32"]:
    print(f"{chi_so:<38}" + "".join(f"{do[ten][chi_so]:>10.3f}" for ten in do) + "  (đo trên máy này)")
print(f"lớp kém nhất của bản gốc: {lop_kem}")

In [ ]:
for ten in ("INT8", "INT4"):
    cong = {
        f"top-1 giảm dưới {NGUONG_TOP1} điểm": do["FP32"]["top-1 (%)"] - do[ten]["top-1 (%)"] < NGUONG_TOP1,
        f"ECE tăng không quá {NGUONG_ECE}": do[ten]["ECE"] - do["FP32"]["ECE"] <= NGUONG_ECE,
        **{f"{lc} giảm không quá {NGUONG_BIEN} điểm": do["FP32"][f"{lc} (%)"] - do[ten][f"{lc} (%)"] <= NGUONG_BIEN
           for lc in LAT_CAT},
    }
    print(f"{ten}: " + "; ".join(f"{k}: {'đạt' if v else 'KHÔNG đạt'}" for k, v in cong.items()) + " (đo trên máy này)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
with torch.no_grad():
    for (ten, m), mau in zip(MO_HINH.items(), ("#2a78d6", "#1baf7a", "#eb6834")):
        if ten == "INT8":
            continue  # gần trùng với FP32, xem bảng ở trên
        logit = m(X_kt)
        for T, kieu, nhan in ((1.0, "o-", "trước co giãn"), (do[ten]["T"], "s--", f"sau co giãn, T = {do[ten]['T']:.2f}")):
            tin, doan = torch.softmax(logit / T, 1).max(1)
            bien_k = torch.linspace(0.3, 1, 15)
            x_, y_ = [], []
            for a, b in zip(bien_k[:-1], bien_k[1:]):
                trong = (tin > a) & (tin <= b)
                if trong.sum() >= 30:
                    x_.append(tin[trong].mean().item())
                    y_.append((doan[trong] == Y_kt[trong]).float().mean().item())
            ax1.plot(x_, y_, kieu, color=mau, ms=4, lw=1.5, alpha=1 if T == 1.0 else 0.7, label=f"{ten}, {nhan}")
ax1.plot([0.3, 1], [0.3, 1], color="0.5", lw=1, ls=":", label="hiệu chuẩn hoàn hảo")
ax1.set_xlabel("độ tự tin trung bình trong khoảng")
ax1.set_ylabel("tỉ lệ đúng trong khoảng")
ax1.set_title("Biểu đồ độ tin cậy (đo trên máy này)")
ax1.legend(loc="upper left", fontsize=8)
nhom = ["tập kiểm tra"] + list(LAT_CAT)
vi_tri = np.arange(len(nhom))
for j, (ten, mau) in enumerate(zip(do, ("#2a78d6", "#1baf7a", "#eb6834"))):
    gia = [do[ten]["top-1 (%)"]] + [do[ten][f"{lc} (%)"] for lc in LAT_CAT]
    ax2.bar(vi_tri + (j - 1) * 0.27, gia, 0.27, color=mau, label=ten)
ax2.set_xticks(vi_tri, nhom, fontsize=8)
ax2.set_ylim(0, 105)
ax2.set_ylabel("độ chính xác (%)")
ax2.set_title("Con số tổng và ba lát cắt (đo trên máy này)")
ax2.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Đọc bảng và các cổng theo thứ tự đã đặt. Trên tập kiểm tra đầy đủ, bản INT8 thường gần như trùng với bản gốc,
còn bản INT4 lệch rõ hơn. Một bản có thể qua cổng top-1 mà trượt cổng hiệu chuẩn hay một lát cắt, và đó đúng là
điều bảng 18 cảnh báo. Hãy xem dòng "lớp kém nhất": con số tổng của FashionMNIST che một lớp kém hơn hẳn, đúng
kiểu thất bại ở nhóm con mà đánh giá tách theo nhóm tìm ra.

Bốn dòng "tự động" áp quy tắc của sách, chỉ hành động khi độ tự tin trên 85 phần trăm, lên độ tự tin trước và
sau co giãn. Với một mô hình hiệu chuẩn tốt, tỉ lệ đúng trong phần được tự động phải không thấp hơn ngưỡng ấy.
Một mô hình quá tự tin đẩy thêm ảnh qua ngưỡng, nên hệ thống tự hành động trên nhiều ảnh hơn mức nó đáng được tin.

Ba lát cắt thấp hơn tập kiểm tra rất nhiều, vì mạng này chưa từng thấy ảnh xoay, bị che hay tối. Sách ước
trường hợp biên chiếm khoảng 5 phần trăm đầu vào thật, nên chúng gần như không đổi con số tổng, mà vẫn là nơi
hệ thống thất bại. Co giãn nhiệt độ, chỉ với một hằng số, kéo đường độ tin cậy về sát đường chéo và hạ ECE,
nhưng không đổi dự đoán nào, nên nó sửa hiệu chuẩn mà không sửa lát cắt.

## Thử thêm

1. Ở phần 2, đổi `ALPHA` thành 0.1. Công suất hiệu dụng giờ gần phần nào hơn, và pin giả định dùng được bao lâu?
2. Ở phần 5, đổi `NGUONG_ECE` thành 0.005 hoặc 0.02. Bản INT4 đổi kết luận ở cổng nào?
3. Ở phần 5, đổi bit của bản thứ hai từ 4 thành 3 trong `MO_HINH`, rồi chạy lại từ ô ấy. Cổng nào trượt đầu
   tiên?

## Tóm lại

* Năng lượng là công suất đo được nhân thời gian: 300 W trong 10 ms là 3 J, khoảng 0.0008 Wh. TDP không thay
  được phép đo ấy.
* Công suất trung bình đi theo chu kỳ hoạt động: 1 phần trăm ở 2 W cộng 99 phần trăm ở 50 mW chỉ là khoảng 69.5
  mW. Một tỉ số TOPS/W chỉ có nghĩa khi tử số và mẫu số được đọc ở cùng một điểm vận hành.
* Theo mô hình 45 nm, INT8 giảm MobileNetV2 từ 4,463 xuống 681 µJ, tức 6.6 lần; khi tính thêm kích hoạt, cả
  con số tuyệt đối lẫn tỉ lệ đều đổi theo giả định, nên tuyên bố về pin cần phép đo cả thiết bị.
* Nghiệm thu một bản nén cần ngưỡng đặt trước cho độ chính xác tổng, hiệu chuẩn và các lát cắt then chốt. Con số
  tổng che lớp kém nhất và các trường hợp biên; co giãn nhiệt độ sửa hiệu chuẩn, không sửa lát cắt.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Đo chuẩn hiệu năng — khi lời hứa tối ưu gặp số đo](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch12-benchmarking/), dựng từ chương
[*Benchmarking*](https://mlsysbook.ai/vol1/benchmarking/benchmarking.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.